# Rung 2 follow-up — the protocol-ablation phase diagram (issue #168)

The positive control grokked with a **sparse** circuit (2026-10-14), so the boundary
question is answerable by ablation instead of speculation: starting from the
Nanda-faithful setup, switch on **one deviation at a time** (minibatch, default
betas, cosine LR, embedding renorm, LayerNorm, grad clip) plus the weight-decay axis.

The sweep runner delegates to exp2's own code, budgets in optimizer steps, and records
the post-audit energy metric; every cell emits a citable per-cell manifest
(`--emit-manifests results/`). Cells already completed are skipped on re-run, and the
working `.pt` manifest carries the in-progress state.

**Before running:** GPU T4/P100 + Internet On. Total ~1.5-2h for the core config on a T4.
If the session dies mid-sweep, re-run the cell: completed cells are skipped and the
remaining ones continue.

In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifests' git_sha is real
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
# The GPU images ship the CUDA torch stack plus numpy, matplotlib, tqdm, pyyaml;
# `uv sync --frozen` would re-resolve ~5GB of CUDA wheels into the limited working
# disk and die mid-download. Fill only real gaps; the manifests record the versions used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}
import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])

In [ ]:
# The sweep. Resumable: re-run this cell after a disconnect — completed cells are
# skipped (the working manifest records them) and the rest continue.
!python -m scripts.phase_diagram_sweep --config configs/phase_diagram_core.yaml --resume --emit-manifests results/

In [ ]:
# Which cells reached the sparse regime, by the post-audit energy criterion?
import json
from pathlib import Path

n_sparse = 0
n_total = 0
for p in sorted(Path('results').glob('phase_diagram_*.json')):
    m = json.load(open(p))
    n_total += 1
    cell = m['args']
    a = m['aggregate']
    k90 = a.get('k_energy_90', {}).get('mean')
    sparse = a.get('energy_sparse', {}).get('mean')
    val = a.get('final_val_acc', {}).get('mean')
    if sparse:
        n_sparse += 1
    print(f"{cell['cell_id']:<22} P={cell['modulus']:<4} wd={cell['weight_decay']:<4} "
          f"val={val:.4f} k_energy_90={k90} sparse={sparse}")
print(f"\nsparse cells: {n_sparse}/{n_total} (energy criterion)")

In [ ]:
import glob

manifests = sorted(glob.glob('results/phase_diagram_*.json'))
print('manifests:', len(manifests))
!zip -j phase_sweep_results.zip $manifests
try:
    from google.colab import files
    files.download('phase_sweep_results.zip')
except ImportError:
    print('Kaggle: the zip is in the notebook output (/kaggle/working).')

## After downloading

1. Pin: `python scripts/pin_remote_run.py --from <dir or zip>` (fail-closed: provenance, seeds/aggregate contract).
2. Cite the per-cell manifests in `portfolio/RESULTS.md` with `<!-- manifest: results/phase_diagram_<cell>.json -->` and run `make verify-claims`.
3. Report the boundary as measured — including every cell that stayed dense.